# Test-Time Scaling on FinQA: Initial Analysis

Loads experiment JSON files from `results/`, plots accuracy vs. number of samples (N) and vs. total generated tokens, and surfaces individual trajectories for failure analysis.

This notebook only reads result files — it does not run any model or execute any generated code itself.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("../results")

In [ ]:
def load_experiments(results_dir: Path) -> list[dict]:
    experiments = []
    for path in sorted(results_dir.glob("*.json")):
        with open(path) as f:
            experiments.append(json.load(f))
    return experiments


experiments = load_experiments(RESULTS_DIR)
print(f"Loaded {len(experiments)} experiment files.")

In [ ]:
summary = pd.DataFrame([
    {
        "experiment_id": e["experiment_id"],
        "strategy": e["strategy"],
        "num_samples": e["num_samples"],
        "model": e["model"],
        "accuracy": e["metrics"]["accuracy"],
        "average_tokens": e["metrics"]["average_tokens"],
        "average_latency": e["metrics"]["average_latency"],
        "num_examples": e["metrics"]["num_examples"],
    }
    for e in experiments
])
summary.sort_values("num_samples")

## Accuracy vs. number of samples (N)

In [ ]:
by_n = summary.sort_values("num_samples")

fig, ax = plt.subplots()
ax.plot(by_n["num_samples"], by_n["accuracy"], marker="o")
ax.set_xscale("log", base=2)
ax.set_xticks(by_n["num_samples"])
ax.set_xticklabels(by_n["num_samples"])
ax.set_xlabel("Number of samples (N)")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs. inference-time compute (N)")
ax.set_ylim(0, 1)
plt.show()

## Accuracy vs. total generated tokens

In [ ]:
by_tokens = summary.sort_values("average_tokens")

fig, ax = plt.subplots()
ax.plot(by_tokens["average_tokens"], by_tokens["accuracy"], marker="o")
ax.set_xlabel("Average total tokens per question")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs. token compute")
ax.set_ylim(0, 1)
plt.show()

## Per-question trajectory inspection

Load the per-example records from a single experiment file for qualitative failure analysis.

In [ ]:
def examples_df(experiment: dict) -> pd.DataFrame:
    rows = []
    for ex in experiment["examples"]:
        answers = [
            t["verification"]["answer"]
            for t in ex["trajectories"]
            if t.get("verification") and t["verification"].get("success")
        ]
        rows.append({
            "question_id": ex["question_id"],
            "correct": ex["correct"],
            "final_answer": ex["final_answer"],
            "num_trajectories": len(ex["trajectories"]),
            "num_successful_executions": len(answers),
            "distinct_answers": sorted(set(answers)),
            "all_failed": len(answers) == 0,
            "disagreement": len(set(answers)) > 1,
        })
    return pd.DataFrame(rows)


# Point this at the N=1 and N=8 (or highest-N) experiment files to compare.
n1_experiments = [e for e in experiments if e["num_samples"] == 1]
n8_experiments = [e for e in experiments if e["num_samples"] >= 8]

n1_df = examples_df(n1_experiments[0]) if n1_experiments else None
n8_df = examples_df(n8_experiments[0]) if n8_experiments else None
n1_df, n8_df

### Cases where N=1 fails but higher N succeeds

In [ ]:
if n1_df is not None and n8_df is not None:
    merged = n1_df.merge(n8_df, on="question_id", suffixes=("_n1", "_nhigh"))
    flips_to_correct = merged[(~merged["correct_n1"]) & (merged["correct_nhigh"])]
    print(f"{len(flips_to_correct)} / {len(merged)} questions flipped from incorrect (N=1) to correct (higher N)")
    flips_to_correct[["question_id", "final_answer_n1", "final_answer_nhigh"]]

### Cases where all samples failed to execute or agree

In [ ]:
if n8_df is not None:
    print("All samples failed execution:")
    display(n8_df[n8_df["all_failed"]][["question_id", "num_trajectories"]])

    print("Samples disagreed on the answer:")
    display(n8_df[n8_df["disagreement"]][["question_id", "distinct_answers"]])

### Execution catching a bad solution (model stated an answer, but its code errored or disagreed)

In [ ]:
if n8_experiments:
    for ex in n8_experiments[0]["examples"][:20]:
        for t in ex["trajectories"]:
            v = t.get("verification")
            if v and not v["success"] and t.get("stated_answer"):
                print(ex["question_id"], "stated:", t["stated_answer"], "| execution_error:", v["execution_error"])

### % of examples where additional samples changed the outcome

In [ ]:
if n1_df is not None and n8_df is not None:
    merged = n1_df.merge(n8_df, on="question_id", suffixes=("_n1", "_nhigh"))
    changed = merged["correct_n1"] != merged["correct_nhigh"]
    print(f"{changed.mean() * 100:.1f}% of examples changed outcome between N=1 and higher N")